<div align="center">

# Start here: The computer, Jupyter and the first lines of Python

**Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists**<br>
Turkish Society of Cardiology · Prof. Dr. Utku Kose

[Course home](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture) · [Setup](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/SETUP.md) · [Week 1](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/tree/main/en/week-01) · [Open in Colab (fallback)](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/start-here/NB00_start_here.ipynb)

</div>

This Jupyter notebook prepares the computer for the course in about 30 minutes. It checks the Python packages, finds the course folder, introduces Jupyter and the first lines of Python with clinical numbers, and draws a first electrocardiogram (ECG). It also sends a first prompt to an artificial intelligence (AI) assistant and checks the code that comes back. The last cell prints **Ready for Week 1** when everything works.

**How to use it.** Each grey box is a code cell. Click into a cell and press Shift+Enter to run it, from top to bottom. A cell that is running shows `[*]` on its left, and a finished cell shows a number. *Run → Run All Cells* runs the whole notebook at once.

## 1. Are the packages installed?

The cell below imports the packages of Week 1 one by one and prints their versions. NumPy, pandas, SciPy and Matplotlib are required and come with Anaconda. The other three are optional: The notebooks of the course use them when they are installed and a built-in fallback when they are not.

In [ ]:
import importlib
import platform

print("Python", platform.python_version())
REQUIRED = ["numpy", "pandas", "scipy", "matplotlib"]
OPTIONAL = ["ipywidgets", "wfdb", "neurokit2"]
missing = []
for name in REQUIRED + OPTIONAL:
    try:
        module = importlib.import_module(name)
        print(f"{name:12s} {getattr(module, '__version__', '')}")
    except ImportError:
        if name in REQUIRED:
            missing.append(name)
            print(f"{name:12s} MISSING: install Anaconda again or run: pip install {name}")
        else:
            print(f"{name:12s} not installed (optional)")

## 2. Where is the course folder?

The notebooks of the course use the course package `vclinic` and the data files of the folder `data`. The cell below looks for the course folder, starting from the folder of this notebook. If the notebook was opened on its own, for example in Google Colab, it downloads the package and the data from GitHub once.

In [ ]:
import pathlib
import sys
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/"
COURSE_FILES = [
    "vclinic/__init__.py", "vclinic/ecg.py", "vclinic/cohort.py",
    "vclinic/measure.py", "vclinic/physionet.py", "vclinic/datasets.py",
    "vclinic/teach.py",
]


def find_course_root():
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "vclinic" / "__init__.py").exists():
            return folder
    return None


ROOT = find_course_root()
if ROOT is None:
    ROOT = pathlib.Path.cwd() / "ai-cardiology-course"
    for rel in COURSE_FILES:
        target = ROOT / rel
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(REPO_RAW + rel, target)
    print("Course package downloaded to", ROOT)
else:
    print("Course folder:", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

## 3. Jupyter in five minutes

A Jupyter notebook mixes text cells, like this one, with code cells. Python keeps everything that a cell creates in memory. The order in which cells run therefore matters: A cell can use a name only after the cell that creates it has run. If the notebook gets confused, for example after cells were run out of order, *Kernel → Restart Kernel and Run All Cells* starts again from a clean state.

Three habits help from the first day. The Tab key completes names, so `pri` followed by Tab offers `print`. A question mark after a name shows its help, for example `print?`. Saving with Ctrl+S (Cmd+S on macOS) keeps the notebook with its results.

## 4. The first lines of Python

A variable is a name for a value. The cell below stores the weight and the height of a patient and computes the body mass index, $\text{BMI} = \text{weight} / \text{height}^2$, in kg/m². The operator `**` raises to a power, and `round(x, 1)` rounds to one decimal.

In [ ]:
weight_kg = 82
height_m = 1.76
bmi = weight_kg / height_m ** 2
print("BMI:", round(bmi, 1), "kg/m²")

The small function below checks the answers of the two exercises. It prints *correct* when an answer lies close enough to the expected value.

In [ ]:
def check(name, answer, expected, tolerance):
    if answer is None:
        print(f"{name}: write your answer first.")
    elif abs(answer - expected) <= tolerance:
        print(f"{name}: correct.")
    else:
        print(f"{name}: not yet. Check the formula and the units.")

**Exercise 0.1.** The mean arterial pressure is often estimated as $\text{MAP} = \text{DBP} + (\text{SBP} - \text{DBP})/3$. Compute it for a blood pressure of 121/71 mmHg: Replace `None` with the formula, using the variables `sbp` and `dbp`.

In [ ]:
sbp = 121
dbp = 71
mean_arterial_pressure = None  # replace None with the formula
check("Exercise 0.1", mean_arterial_pressure, 87.67, 0.05)

**Exercise 0.2.** The heart rate in beats per minute (bpm) is 60 divided by the RR interval in seconds. Compute it for an RR interval of 706 ms. Mind the unit: 706 ms is 0.706 s.

In [ ]:
rr_ms = 706
heart_rate_bpm = None  # replace None with the formula
check("Exercise 0.2", heart_rate_bpm, 84.99, 0.05)

In [ ]:
#@title Solutions (open after trying)
mean_arterial_pressure = dbp + (sbp - dbp) / 3
heart_rate_bpm = 60 / (rr_ms / 1000)
print(round(mean_arterial_pressure, 2), "mmHg;", round(heart_rate_bpm, 2), "bpm")

## 5. A first look at an ECG

The course package `vclinic` creates the patients of the virtual clinic. The cell below creates the registry of 500 synthetic patients, takes patient VC-0317, the case of Week 1, and draws ten seconds of lead II on ECG paper. Week 1 explains every step of this picture.

In [ ]:
import matplotlib.pyplot as plt

import vclinic
from vclinic import teach

teach.set_language("en")
clinic = vclinic.make_cohort(500)
case = vclinic.patient_ecg(clinic, "VC-0317")
print(case)

fig, ax = plt.subplots(figsize=(12, 3))
teach.plot_strip(ax, case.time, case.lead("II"), label="II")
plt.show()

## 6. Your first prompt

The course uses AI assistants to write code from Week 1, under one strict rule: A prompt describes the data, never the patient. Every exercise of the course can also be solved on the [prompt path](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/PROMPT_PATH.md): A prompt goes to an assistant, and the code that comes back is run and checked. Each prompt is written on the prompt card of the course, which has seven fields.

Copy the prompt below into an AI assistant, for example Anaconda Assistant in JupyterLab or Gemini in Colab. Paste the code it returns into the next cell and run it. The cell after it checks the function with a case that the prompt does not mention. Without an assistant, open the solution cell and read it line by line: Python step 1 of Week 1 explains functions.

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook with Python 3, in which NumPy can be imported as np.
Data: R-peak positions of one ECG lead as sample indices in a list, and the sampling frequency fs in Hz.
Task: Write a function mean_heart_rate(peaks, fs) that returns the mean heart rate in beats per minute.
The RR intervals in seconds are the differences of successive peak positions divided by fs, and the
heart rate is 60 divided by their mean.
Constraints: Use only NumPy. Do not load files and do not install anything.
Output: Return only the code in one block, with a short comment above each step.
Test: Check with an assert statement that peaks at 0, 360 and 720 at 360 Hz give 60 bpm.
</pre>

In [ ]:
# Paste the code from the AI assistant here and run it.

In [ ]:
# Check of the first prompt with a case that the prompt does not mention
try:
    answer = mean_heart_rate([0, 300, 600, 900], 500)
except NameError:
    answer = None
check("First prompt", answer, 100.0, 0.05)

In [ ]:
#@title Solution (open after trying)
import numpy as np


def mean_heart_rate(peaks, fs):
    # RR intervals in seconds
    rr_s = np.diff(peaks) / fs
    # mean heart rate in beats per minute
    return 60 / rr_s.mean()


assert round(mean_heart_rate([0, 360, 720], 360), 1) == 60.0
print(round(mean_heart_rate([0, 300, 600, 900], 500), 1), "bpm")

**Explain and log.** Ask the assistant to explain its code line by line, then explain it once more in your own words. The check used peaks every 300 samples at 500 Hz: The RR interval is 300 / 500 = 0.6 s, and the heart rate is 60 / 0.6 = 100 bpm. Code is accepted when its result passes such a check and every line can be explained. Week 1 adds the code safety checklist and the AI usage log, where each use of an assistant is recorded. A prompt never contains names, dates of birth, hospital numbers or recordings of real patients.

## 7. Ready for Week 1?

The last cell collects the checks of this notebook.

In [ ]:
data_ok = (ROOT / "vclinic" / "__init__.py").exists()
if missing:
    print("Install the missing packages first:", ", ".join(missing))
elif not data_ok:
    print("The course folder was not found. See SETUP.md.")
else:
    print("vclinic", vclinic.__version__, "·", "course folder", ROOT)
    print("Ready for Week 1.")

**Next:** Open the [Week 1 page](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/tree/main/en/week-01) and the Jupyter notebook `en/week-01/NB01_python_and_ecg_basics.ipynb`.

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University<br>
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. First version: November 2026. Content CC BY 4.0, code MIT.*

</div>